In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("anime-dataset-2023.csv")

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
df["Scored By"] = pd.to_numeric(df["Scored By"], errors="coerce")

In [ ]:
print(df["Score"].isna().sum())
print(df["Scored By"].isna().sum())

In [ ]:
df["Score_missing"] = df["Score"].isna().astype(int)

df["Score"] = df["Score"].fillna(df["Score"].median())
df["Scored By"] = df["Scored By"].fillna(0)

In [ ]:
df["Scored By"].describe()

In [ ]:
R = df["Score"]
v = df["Scored By"]
C = df["Score"].mean()
m = df["Scored By"].quantile(0.75)

In [ ]:
df["weighted_score"] = (R*v + C*m)/(v+m)

print(df[["Name","weighted_score"]].head())

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["Popularity"] = df["Popularity"].max() - df["Popularity"]

cols = ["Popularity", "Members", "Favorites"]

df[cols] = scaler.fit_transform(df[cols])

In [ ]:
df["weighted_score"] = scaler.fit_transform(df[["weighted_score"]])

df["final_score"] = (
    0.85 * df["weighted_score"] +
    0.15 * df["Popularity"]
)

In [ ]:
revised_df = df[["Name","Score","Scored By","weighted_score","Popularity","Members","Favorites","final_score"]].copy()

final_df = revised_df.sort_values(by = "final_score", ascending= False)

final_df.head(10)

In [ ]:
print(len(df[df["Genres"] == "UNKNOWN"]))

In [ ]:
genre_df = df[df["Genres"].str.upper() != "UNKNOWN"].copy()

In [ ]:
def get_unique_options(column):
    values = (
        df.loc[df[column].str.upper() != "UNKNOWN", column]
        .dropna()
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
    )
    values = values[values != ""]
    return sorted(values.unique(), key=str.lower)


unique_genres = get_unique_options("Genres")
unique_types = get_unique_options("Type")
unique_studios = get_unique_options("Studios")
unique_ratings = get_unique_options("Rating")

print("Genres:", unique_genres)
print("Total genres:", len(unique_genres))

print("\nTypes:", unique_types)
print("Total types:", len(unique_types))

print("\nStudios sample:", unique_studios[:20])
print("Total studios:", len(unique_studios))

print("\nRatings:", unique_ratings)
print("Total ratings:", len(unique_ratings))

In [ ]:
from rapidfuzz import process, fuzz

def get_best_genre_match(genre, unique_genres, score_cutoff=70):
    genre = genre.lower().strip()

    # Create a mapping of lowercase genre -> original genre
    genre_map = {g.lower(): g for g in unique_genres}

    # Exact match
    if genre in genre_map:
        return genre_map[genre]

    # Fuzzy match
    match = process.extractOne(
        genre,
        genre_map.keys(),
        scorer=fuzz.WRatio,
        score_cutoff=score_cutoff
    )

    if match:
        matched_genre, score, _ = match
        print(f"Genre found. Using '{genre_map[matched_genre]}' ({score:.1f}% match)")
        return genre_map[matched_genre]

    return None

In [ ]:
genre_df["Genres"] = (
    genre_df["Genres"]
    .str.lower()
    .str.split(",")
    .apply(lambda genres: [g.strip() for g in genres])
)

In [ ]:
genre_df["Genre_Set"] = genre_df["Genres"].apply(set)

In [ ]:
def _as_list(values):
    if values is None:
        return []
    if isinstance(values, str):
        return [values]
    return list(values)


def _split_values(value):
    if pd.isna(value):
        return []
    return [item.strip().lower() for item in str(value).split(",") if item.strip()]


def _is_known(value):
    return pd.notna(value) and str(value).strip().upper() != "UNKNOWN"


def _unique_metadata_values(column):
    return get_unique_options(column)


def _best_metadata_match(value, choices, label, score_cutoff=70):
    value = value.lower().strip()
    choice_map = {choice.lower(): choice for choice in choices}

    if value in choice_map:
        return choice_map[value]

    match = process.extractOne(
        value,
        choice_map.keys(),
        scorer=fuzz.WRatio,
        score_cutoff=score_cutoff
    )

    if match:
        matched_value, score, _ = match
        print(f"{label} found. Using '{choice_map[matched_value]}' ({score:.1f}% match)")
        return choice_map[matched_value]

    return None


def _match_inputs(values, choices, label):
    matched = []
    for value in _as_list(values):
        match = _best_metadata_match(str(value), choices, label)
        if match:
            matched.append(match.lower().strip())
    return set(matched)


def _metadata_recommend(column, values, top_n=10, label=None):
    label = label or column
    choices = _unique_metadata_values(column)
    selected = _match_inputs(values, choices, label)

    if not selected:
        print(f"No valid {label.lower()} found.")
        return None

    recommendations = df[df[column].apply(_is_known)].copy()
    recommendations[f"{column.lower()}_set"] = recommendations[column].apply(lambda value: set(_split_values(value)))
    recommendations["metadata_score"] = recommendations[f"{column.lower()}_set"].apply(
        lambda row_values: len(selected & row_values) / len(selected)
    )
    recommendations = recommendations[recommendations["metadata_score"] > 0]
    recommendations["recommendation_score"] = (
        0.7 * recommendations["metadata_score"] +
        0.3 * recommendations["final_score"]
    )

    top_recommendations = recommendations.nlargest(top_n, "recommendation_score")
    for _, row in top_recommendations.iterrows():
        print(f"{row['Name']} {row['Image URL']}")


def recommend_by_genres(genres, top_n=10):
    return _metadata_recommend("Genres", genres, top_n, label="Genre")


def recommend_by_type(type_name, top_n=10):
    return _metadata_recommend("Type", type_name, top_n, label="Type")


def recommend_by_studios(studios, top_n=10):
    return _metadata_recommend("Studios", studios, top_n, label="Studio")


def recommend_by_rating(rating, top_n=10):
    return _metadata_recommend("Rating", rating, top_n, label="Rating")


def hybrid_metadata_recommend(genres=None, type_name=None, studios=None, rating=None, top_n=10):
    filters = {
        "Genres": _match_inputs(genres, _unique_metadata_values("Genres"), "Genre"),
        "Type": _match_inputs(type_name, _unique_metadata_values("Type"), "Type"),
        "Studios": _match_inputs(studios, _unique_metadata_values("Studios"), "Studio"),
        "Rating": _match_inputs(rating, _unique_metadata_values("Rating"), "Rating"),
    }
    filters = {column: selected for column, selected in filters.items() if selected}

    if not filters:
        print("No valid metadata filters found.")
        return None

    recommendations = df.copy()
    metadata_score_columns = []

    for column, selected in filters.items():
        score_column = f"{column.lower()}_match_score"
        metadata_score_columns.append(score_column)
        recommendations[score_column] = recommendations[column].apply(
            lambda value: len(selected & set(_split_values(value))) / len(selected)
        )

    recommendations["metadata_score"] = recommendations[metadata_score_columns].mean(axis=1)
    recommendations = recommendations[recommendations["metadata_score"] > 0]
    recommendations["recommendation_score"] = (
        0.7 * recommendations["metadata_score"] +
        0.3 * recommendations["final_score"]
    )

    top_recommendations = recommendations.nlargest(top_n, "recommendation_score")
    for _, row in top_recommendations.iterrows():
        print(f"{row['Name']} {row['Image URL']}")

In [ ]:
recommend_by_genres(["romance"], 20)

In [ ]:
recommend_by_type("TV", 20)

In [ ]:
recommend_by_studios("Kyoto Animation", 50)